# Image Processing

This notebook introduces a compact image-processing workflow using NumPy, OpenCV, and Matplotlib. It uses a generated sample image, so it runs without downloading any external data.

## 1. Imports

OpenCV stores color images as BGR by default. Matplotlib expects RGB, so the helper below converts images before display.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np


def show_image(title, image, cmap=None):
    plt.figure(figsize=(6, 4))
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

## 2. Load an Image

In [ ]:
im = cv2.imread("../seals-dataset/ground_truth/01_510_800_820_280_114/10_3258896_1.png")

show_image("Seal image", im)

## 3. Inspect Pixels and Channels

Images are arrays. A color image has height, width, and channel dimensions.

In [ ]:
print(f"Shape: {im.shape}")
print(f"Data type: {im.dtype}")
print(f"Min pixel value: {im.min()}")
print(f"Max pixel value: {im.max()}")

blue, green, red = cv2.split(im)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, channel, title in zip(axes, [blue, green, red], ["Blue", "Green", "Red"]):
    ax.imshow(channel, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## 4. Convert to Grayscale

Many classic image-processing operations start with a single intensity channel.

In [ ]:
gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
show_image("Grayscale", gray, cmap="gray")

## 5. Reduce Noise

Gaussian blur smooths local variation. It is often used before thresholding or edge detection.

In [ ]:
blurred = cv2.GaussianBlur(gray, (7, 7), sigmaX=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Original grayscale")
axes[0].axis("off")
axes[1].imshow(blurred, cmap="gray")
axes[1].set_title("Blurred")
axes[1].axis("off")
plt.tight_layout()

## 6. Threshold the Image

Thresholding converts a grayscale image into a binary mask. Otsu's method estimates a threshold from the image histogram.

In [ ]:
threshold_value, mask = cv2.threshold(
    blurred,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU,
)

print(f"Otsu threshold: {threshold_value:.1f}")
show_image("Binary mask", mask, cmap="gray")

## 7. Detect Edges

Canny edge detection finds strong intensity changes. It is sensitive to blur and threshold choices.

In [ ]:
edges = cv2.Canny(blurred, threshold1=60, threshold2=160)
show_image("Canny edges", edges, cmap="gray")

## 8. Detect Components

Connected-component detection groups neighboring foreground pixels into separate objects. After cleaning the binary mask, each component gets an area, bounding box, and centroid.

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
component_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
component_mask = cv2.morphologyEx(component_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    component_mask,
    connectivity=8,
)

min_area = 500
components = []
component_overlay = im.copy()

for label_id in range(1, num_labels):
    x = stats[label_id, cv2.CC_STAT_LEFT]
    y = stats[label_id, cv2.CC_STAT_TOP]
    w = stats[label_id, cv2.CC_STAT_WIDTH]
    h = stats[label_id, cv2.CC_STAT_HEIGHT]
    area = stats[label_id, cv2.CC_STAT_AREA]
    cx, cy = centroids[label_id]

    if area < min_area:
        continue

    components.append(
        {
            "label": label_id,
            "area": int(area),
            "bbox": (int(x), int(y), int(w), int(h)),
            "centroid": (float(cx), float(cy)),
        }
    )

components = sorted(components, key=lambda item: item["area"], reverse=True)

for index, component in enumerate(components, start=1):
    x, y, w, h = component["bbox"]
    cx, cy = component["centroid"]
    cv2.rectangle(component_overlay, (x, y), (x + w, y + h), (0, 255, 255), 3)
    cv2.circle(component_overlay, (int(cx), int(cy)), 5, (0, 0, 255), thickness=-1)
    cv2.putText(
        component_overlay,
        str(index),
        (x, max(20, y - 8)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2,
        cv2.LINE_AA,
    )

print(f"Detected {len(components)} components with area >= {min_area} pixels")
for index, component in enumerate(components[:10], start=1):
    print(
        f"{index:>2}: area={component['area']:>6}, "
        f"bbox={component['bbox']}, "
        f"centroid=({component['centroid'][0]:.1f}, {component['centroid'][1]:.1f})"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(component_mask, cmap="gray")
axes[0].set_title("Cleaned component mask")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(component_overlay, cv2.COLOR_BGR2RGB))
axes[1].set_title("Detected components")
axes[1].axis("off")
plt.tight_layout()

## 9. Save Results

Saving intermediate results makes notebooks easier to audit and reuse.

In [ ]:
output_dir = Path("output/notebooks/image_processing_seals")
output_dir.mkdir(parents=True, exist_ok=True)

cv2.imwrite(str(output_dir / "sample_original.png"), im)
cv2.imwrite(str(output_dir / "grayscale.png"), gray)
cv2.imwrite(str(output_dir / "mask.png"), mask)
cv2.imwrite(str(output_dir / "edges.png"), edges)
cv2.imwrite(str(output_dir / "component_mask.png"), component_mask)
cv2.imwrite(str(output_dir / "component_overlay.png"), component_overlay)

sorted(path.name for path in output_dir.iterdir())